# Day 33 Tutorial：GCN、GraphSAGE 与 GAT

## Goal

在相同 KarateClub 开发协议和三种配对种子下比较 GCN、GraphSAGE、GAT，并从预先指定的 GAT 读取最多 8 条第一层注意力记录。

**解释边界：** 注意力权重是模型内部关联权重，不是因果证据；test 标签保持封存。

## Setup

需要 `torch`、`torch_geometric`、`numpy` 与 `pandas`。本 Notebook 不自动安装依赖、不隐藏报错、不持久化墙钟时间。GAT 使用 2 个头，每头宽度 8，拼接后的隐藏总宽度仍为 16。

In [1]:
import random

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from torch_geometric.datasets import KarateClub
from torch_geometric.nn import GATConv, GCNConv, SAGEConv

MASK_SEED = 20260728
SEEDS = [7, 17, 27]
HIDDEN_CHANNELS = 16
GAT_HEADS = 2
EPOCHS = 120
DROPOUT = 0.5
LEARNING_RATE = 0.01
WEIGHT_DECAY = 5e-4

assert HIDDEN_CHANNELS % GAT_HEADS == 0

## Steps

### 1. 建立与前两天相同的数据和掩码

训练掩码来自数据集；其余节点用固定掩码种子随机平分。模型种子不会改变划分。

In [2]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def make_fixed_masks(graph, mask_seed=MASK_SEED):
    train_mask = graph.train_mask.clone()
    remaining_nodes = (~train_mask).nonzero(as_tuple=False).view(-1)
    generator = torch.Generator().manual_seed(mask_seed)
    order = torch.randperm(
        remaining_nodes.numel(),
        generator=generator,
    )
    remaining_nodes = remaining_nodes[order]
    valid_count = remaining_nodes.numel() // 2

    valid_mask = torch.zeros(graph.num_nodes, dtype=torch.bool)
    test_mask = torch.zeros(graph.num_nodes, dtype=torch.bool)
    valid_mask[remaining_nodes[:valid_count]] = True
    test_mask[remaining_nodes[valid_count:]] = True
    return train_mask, valid_mask, test_mask


dataset = KarateClub()
data = dataset[0]
train_mask, valid_mask, test_mask = make_fixed_masks(data)

print("x:", tuple(data.x.shape))
print("edge_index:", tuple(data.edge_index.shape))
print(
    "mask sizes:",
    {
        "train": int(train_mask.sum()),
        "validation": int(valid_mask.sum()),
        "test_sealed": int(test_mask.sum()),
    },
)

x: (34, 34)
edge_index: (2, 156)
mask sizes: {'train': 4, 'validation': 15, 'test_sealed': 15}


### 2. 定义三个两层模型

GCN/SAGE 第一层直接输出 16 维；GAT 第一层输出 `8×2=16` 维。三个模型使用相同外部 Dropout，GAT 层内部 attention Dropout 固定为 0。

In [3]:
class SmallGCN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)
        self.dropout = dropout

    def forward(self, x, edge_index):
        hidden = F.relu(self.conv1(x, edge_index))
        hidden = F.dropout(
            hidden,
            p=self.dropout,
            training=self.training,
        )
        return self.conv2(hidden, edge_index)


class SmallGraphSAGE(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)
        self.dropout = dropout

    def forward(self, x, edge_index):
        hidden = F.relu(self.conv1(x, edge_index))
        hidden = F.dropout(
            hidden,
            p=self.dropout,
            training=self.training,
        )
        return self.conv2(hidden, edge_index)


class SmallGAT(nn.Module):
    def __init__(
        self,
        in_channels,
        hidden_channels,
        out_channels,
        dropout,
        heads=GAT_HEADS,
    ):
        super().__init__()
        if hidden_channels % heads != 0:
            raise ValueError("hidden_channels 必须能被 heads 整除")
        channels_per_head = hidden_channels // heads
        self.conv1 = GATConv(
            in_channels,
            channels_per_head,
            heads=heads,
            concat=True,
            dropout=0.0,
        )
        self.conv2 = GATConv(
            hidden_channels,
            out_channels,
            heads=1,
            concat=False,
            dropout=0.0,
        )
        self.dropout = dropout

    def forward(self, x, edge_index):
        hidden = F.elu(self.conv1(x, edge_index))
        hidden = F.dropout(
            hidden,
            p=self.dropout,
            training=self.training,
        )
        return self.conv2(hidden, edge_index)


def count_trainable_parameters(model):
    return sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )


MODEL_BUILDERS = {
    "GCN": SmallGCN,
    "GraphSAGE": SmallGraphSAGE,
    "GAT": SmallGAT,
}

### 3. 训练函数与多头 shape 预检

先在训练循环外确认每个模型都输出 `[N,C]`，并确认 GAT 第一层拼接后是 `[N,16]`。

In [4]:
def masked_accuracy(logits, labels, mask):
    predictions = logits.argmax(dim=1)
    return (
        (predictions[mask] == labels[mask])
        .float()
        .mean()
        .item()
    )


def train_one_run(model_class, seed):
    set_seed(seed)
    model = model_class(
        dataset.num_features,
        HIDDEN_CHANNELS,
        dataset.num_classes,
        DROPOUT,
    )
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    for _ in range(EPOCHS):
        model.train()
        optimizer.zero_grad()
        logits = model(data.x, data.edge_index)
        loss = F.cross_entropy(
            logits[train_mask],
            data.y[train_mask],
        )
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        logits = model(data.x, data.edge_index)
        valid_accuracy = masked_accuracy(
            logits,
            data.y,
            valid_mask,
        )
    return model, valid_accuracy, loss.item()


sample_gat = SmallGAT(
    dataset.num_features,
    HIDDEN_CHANNELS,
    dataset.num_classes,
    DROPOUT,
)
sample_gat.eval()
with torch.no_grad():
    sample_hidden = sample_gat.conv1(data.x, data.edge_index)
    sample_logits = sample_gat(data.x, data.edge_index)

print("GAT first hidden:", tuple(sample_hidden.shape))
print("GAT logits:", tuple(sample_logits.shape))

GAT first hidden: (34, 16)
GAT logits: (34, 4)


### 4. 三个模型使用三种配对种子

所有 9 次结果都保留。预先指定 `seed=7` 的 GAT 用于后面的张量预览，而不是先看结果后挑模型。

In [5]:
records = []
trained_gat_for_preview = None

for seed in SEEDS:
    for model_name, model_class in MODEL_BUILDERS.items():
        trained_model, valid_accuracy, final_train_loss = (
            train_one_run(model_class, seed)
        )
        records.append(
            {
                "model": model_name,
                "seed": seed,
                "valid_accuracy": valid_accuracy,
                "final_train_loss": final_train_loss,
                "parameters": count_trainable_parameters(trained_model),
            }
        )
        if model_name == "GAT" and seed == SEEDS[0]:
            trained_gat_for_preview = trained_model

run_table = pd.DataFrame(records)
display(run_table)

,model,seed,valid_accuracy,final_train_loss,parameters
0,GCN,7,0.666667,0.049145,628
1,GraphSAGE,7,0.533333,0.095912,1236
2,GAT,7,0.733333,0.021982,668
3,GCN,17,0.800000,0.067630,628
4,GraphSAGE,17,0.466667,0.001454,1236
5,GAT,17,0.800000,0.022278,668
6,GCN,27,0.866667,0.043884,628
7,GraphSAGE,27,0.333333,0.012318,1236
8,GAT,27,0.866667,0.037155,668


### 5. 汇总 validation mean/std 与参数量

表格不做“冠军”标记。均值必须和波动、参数量、样本量与实验边界一起看。

In [6]:
summary = (
    run_table
    .groupby("model", as_index=False)
    .agg(
        valid_mean=("valid_accuracy", "mean"),
        valid_std=("valid_accuracy", "std"),
        parameters=("parameters", "first"),
        runs=("seed", "count"),
    )
)
display(summary)

,model,valid_mean,valid_std,parameters,runs
0,GAT,0.800000,0.066667,668,3
1,GCN,0.777778,0.101835,628,3
2,GraphSAGE,0.444444,0.101835,1236,3


### 6. 读取第一层注意力并只展示 8 行

返回边数可能因自动加入自环而大于原始边数。这里不排序，前 8 行只是张量结构预览，不是“最重要边”。

In [7]:
trained_gat_for_preview.eval()
with torch.no_grad():
    attention_hidden, (attention_edges, alpha) = (
        trained_gat_for_preview.conv1(
            data.x,
            data.edge_index,
            return_attention_weights=True,
        )
    )

preview_rows = min(8, attention_edges.shape[1])
attention_preview = pd.DataFrame(
    {
        "source": attention_edges[0, :preview_rows].cpu().numpy(),
        "target": attention_edges[1, :preview_rows].cpu().numpy(),
        "head_0": alpha[:preview_rows, 0].cpu().numpy(),
        "head_1": alpha[:preview_rows, 1].cpu().numpy(),
    }
)

print("attention hidden:", tuple(attention_hidden.shape))
print("returned attention edges:", tuple(attention_edges.shape))
print("alpha:", tuple(alpha.shape))
display(attention_preview)

attention hidden: (34, 16)
returned attention edges: (2, 190)
alpha: (190, 2)


,source,target,head_0,head_1
0,0,1,0.012182,0.016450
1,0,2,0.026535,0.012782
2,0,3,0.036374,0.027233
3,0,4,0.107061,0.230322
4,0,5,0.173554,0.141147
5,0,6,0.096825,0.173768
6,0,7,0.054816,0.042074
7,0,8,0.012906,0.017436


### 7. 检查一个目标节点的归一化

对返回列表中的第一个目标节点，检查同一头所有入边权重之和是否接近 1。这只验证 softmax 结构，不是因果检验。

In [8]:
target_node = int(attention_edges[1, 0])
incoming_to_target = attention_edges[1] == target_node
head_sums = alpha[incoming_to_target].sum(dim=0)

print("target node:", target_node)
print("incoming edge count:", int(incoming_to_target.sum()))
print("attention sum by head:", head_sums.cpu().numpy())

target node: 1
incoming edge count: 10
attention sum by head: [1.        1.0000001]


## Checks

检查公平协议、shape、注意力返回结构与有限输出。test 只用于布尔掩码覆盖检查。

In [9]:
assert not torch.any(train_mask & valid_mask)
assert not torch.any(train_mask & test_mask)
assert not torch.any(valid_mask & test_mask)
assert torch.all(train_mask | valid_mask | test_mask)

assert sample_hidden.shape == (data.num_nodes, HIDDEN_CHANNELS)
assert sample_logits.shape == (data.num_nodes, dataset.num_classes)
assert len(run_table) == len(SEEDS) * len(MODEL_BUILDERS)
assert run_table.groupby("model")["seed"].nunique().eq(len(SEEDS)).all()
assert run_table["valid_accuracy"].between(0, 1).all()
assert np.isfinite(
    run_table[["valid_accuracy", "final_train_loss", "parameters"]]
).all().all()

assert attention_hidden.shape == (data.num_nodes, HIDDEN_CHANNELS)
assert attention_edges.shape[0] == 2
assert alpha.shape == (attention_edges.shape[1], GAT_HEADS)
assert len(attention_preview) <= 8
assert torch.allclose(
    head_sums,
    torch.ones_like(head_sums),
    atol=1e-5,
)

print(
    "Checks passed: paired comparison、GAT shape、bounded attention "
    "均正确；test 标签未用于评价。"
)

Checks passed: paired comparison、GAT shape、bounded attention 均正确；test 标签未用于评价。


## Next Steps

1. 完成 `03_exercises.md`，特别是“注意力不是因果”的表述练习。
2. Day 34 将从节点分类转向整图表示与图分类，先明确输出单位发生了变化。
3. 如果未来研究分子图，仍需独立数据划分、传统/ECFP 基线和最终 test。
4. 不把本 Notebook 中的社交图注意力解释成化学键机制。